# 🚘 ClaimGuard AI Kaggle API Node (P100 GPU Edition)
Run this notebook in Kaggle using **P100 GPU** to create a free backend API node with Llama 3 8B. 
It outputs an `ngrok` URL at the bottom. Copy that URL and paste it into the local frontend using the **Connect Node** button.

In [ ]:
!pip install -q fastapi uvicorn python-multipart requests pdf2image pyngrok nest_asyncio
!apt-get update -y
!apt-get install -y poppler-utils tesseract-ocr

In [ ]:
import os
import subprocess
import time
import requests

print("Starting Ollama Server...")
!rm -rf /usr/local/lib/ollama
!rm -rf /usr/local/bin/ollama
!curl -fsSL https://ollama.com/install.sh | sh

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["OLLAMA_NUM_PARALLEL"] = "1"
env["OLLAMA_MAX_VRAM"] = "16384" # P100 Capacity
ollama_process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(10)

print("Pulling llama3 8b model...")
!ollama pull llama3:8b
print("Ollama Ready!")

In [ ]:
import io
import base64
import json
import shutil
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from pdf2image import convert_from_path
import nest_asyncio
import uvicorn
from pyngrok import ngrok

OCR_SPACE_API_KEY = "K87093604688957"

def extract_text_from_file(file_path):
    if not file_path: return ""
    file_path_lower = file_path.lower()
    if file_path_lower.endswith(('.png', '.jpg', '.jpeg')):
        with open(file_path, "rb") as f:
            base64_encoded = base64.b64encode(f.read()).decode('utf-8')
        payload = {'apikey': OCR_SPACE_API_KEY, 'base64Image': f"data:image/jpeg;base64,{base64_encoded}", 'language': 'eng', 'isOverlayRequired': False}
        response = requests.post('https://api.ocr.space/parse/image', data=payload)
        return response.json()['ParsedResults'][0]['ParsedText']
    elif file_path_lower.endswith('.pdf'):
        images = convert_from_path(file_path, dpi=100) 
        full_text = ""
        for img in images:
            img_byte_arr = io.BytesIO()
            img.save(img_byte_arr, format='JPEG', quality=40) 
            base64_encoded = base64.b64encode(img_byte_arr.getvalue()).decode('utf-8')
            payload = {'apikey': OCR_SPACE_API_KEY, 'base64Image': f"data:image/jpeg;base64,{base64_encoded}", 'language': 'eng', 'isOverlayRequired': False}
            response = requests.post('https://api.ocr.space/parse/image', data=payload)
            full_text += response.json()['ParsedResults'][0]['ParsedText'] + "\n\n"
            time.sleep(1)
        return full_text
    return ""

def car_claim_evaluator(policy_text, bill_text):
    sys_prompt = """You are an Automated Car Insurance Claim Evaluation Engine. Analyze the policy and the bill. Determine coverage and final payables using strict rules. Output only standard JSON matching the breakdown format."""
    prompt = f"POLICY:\n{policy_text}\n\nBILL:\n{bill_text}"
    payload = {"model": "llama3:8b", "prompt": prompt, "system": sys_prompt, "stream": False, "format": "json", "options": {"num_gpu": 99}}
    response = requests.post("http://localhost:11434/api/generate", json=payload, timeout=300)
    return json.loads(response.json().get('response', '{}'))

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/api/evaluate")
async def evaluate_claim(policy_file: UploadFile = File(...), bill_file: UploadFile = File(...)):
    os.makedirs("tmp", exist_ok=True)
    p_path = os.path.join("tmp", policy_file.filename)
    b_path = os.path.join("tmp", bill_file.filename)
    with open(p_path, "wb") as f: shutil.copyfileobj(policy_file.file, f)
    with open(b_path, "wb") as f: shutil.copyfileobj(bill_file.file, f)
    try:
        return car_claim_evaluator(extract_text_from_file(p_path), extract_text_from_file(b_path))
    finally:
        os.remove(p_path)
        os.remove(b_path)

# Start Ngrok and Uvicorn
ngrok.set_auth_token("") # OPTIONAL: Enter authtoken if your free tier is capped
public_url = ngrok.connect(8000).public_url
print("\n======================================================")
print(f"\u2705 KAGGLE API NODE LIVE AT: {public_url}")
print("COPY THE URL ABOVE AND CLICK 'CONNECT NODE' ON THE FRONTEND")
print("======================================================\n")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)